### Этап 0. Загрузка библиотек

In [ ]:
import pandas as pd
import numpy as np

import lightgbm as lgb
import catboost as cb
import xgboost as xgb

from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTreeClassifier

CARDINALITY_THRESHOLD = 15          # Порог для разделения категориальных признаков

### Этап 1. Загрузка, анализ и предварительная обработка данных

In [ ]:
df = pd.read_csv('../datasets/training.csv')

In [ ]:
print(df.shape)

In [ ]:
df.head(5)

In [ ]:
df.info()

###  Разбиение данных на обучающую, валидационную и тестовую выборки

In [ ]:
df['PurchDate']=pd.to_datetime(df['PurchDate'], format='%m/%d/%Y')   # Конвертируем PurchDate в datetime  
df_sorted = df.sort_values('PurchDate').reset_index(drop=True)      # Сортируем по дате 


n = len (df_sorted)                                                 # Определяем границы для разбиения по строкам
train_end = n//3
valid_end = 2*n//3

# Разбиваем датафрейм на обучающую, валидационную и тестовую выборки в пропорциях (1/3 1/3 1/3)
df_train = df_sorted.iloc[:train_end].copy()
df_valid = df_sorted.iloc[train_end:valid_end].copy()
df_test = df_sorted[valid_end:].copy()

print(f"Train: {df_train['PurchDate'].min()} - {df_train['PurchDate'].max()} Количество строк - {len(df_train)}")
print(f"Valid: {df_valid['PurchDate'].min()} - {df_valid['PurchDate'].max()} Количество строк - {len(df_valid)}")
print(f"Test: {df_test['PurchDate'].min()} - {df_test['PurchDate'].max()} Количество строк - {len(df_test)}")


###  Предобработка категориальных признаков

In [ ]:
def get_column_types(df, cardinality_threshold=10, binary_as_categorical=True):
    """
    Автоматически определяет категориальные и числовые колонки.
    
    Параметры:
    - cardinality_threshold: макс. число уникальных значений для категории
    - binary_as_categorical: считать ли бинарные признаки (0/1) категориальными
    
    Возвращает:
    - categorical_cols: список категориальных колонок
    - numerical_cols: список числовых колонок
    """
    categorical_cols = []
    numerical_cols = []
    for col in df.columns:
        if col in ['IsBadBuy', 'RefId', 'PurchDate']:       #Пропускаем целевую переменную и идентификаторы
            continue
        if df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col]):  # Если тип объекта или строки — точно категория
            categorical_cols.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):        # Если числовой тип
            n_unique=df[col].nunique()
            if binary_as_categorical and n_unique ==2:      # бинарные значения относим к категориальным признакам
                categorical_cols.append(col)
            elif n_unique<=cardinality_threshold:           # если значений меньше cardinality_threshold=15 то тоже относим к категориальным признакам
                categorical_cols.append(col)
            else:
                numerical_cols.append(col)
    return categorical_cols, numerical_cols



In [ ]:
cat_cols, num_cols = get_column_types(df_train, CARDINALITY_THRESHOLD)

num_cols.extend(['VehYear', 'VehicleAge'])                              #  Переносим ординальные признаки в числовые
cat_cols = [c for c in cat_cols if c not in ['VehYear', 'VehicleAge']]

high_card_cat = ['Model', 'Trim', 'SubModel', 'BYRNO', 'VNZIP1', 'WheelTypeID']
cat_cols = list(set(cat_cols + high_card_cat))
num_cols = [c for c in num_cols if c not in high_card_cat]

print (f"Категориальные признаки ({len(cat_cols)} : {sorted(cat_cols)})")
print (f"Числовые признаки ({len(num_cols)} : {sorted(num_cols)})")

### Заполнение пропусков

In [ ]:
def fill_missing(train, valid, test, c_cols, n_cols):
    '''  
    Функция заполняет отсутствующие данные 
    для категориальных признаков - 'UNKNOWN'
    для числовых данных - медианное значение расчитанное для train
    '''
    for col in c_cols:
        train[col] = train[col].fillna('UNKNOWN')
        valid[col] = valid[col].fillna('UNKNOWN')
        test[col] = test[col].fillna('UNKNOWN')
    for col in n_cols:
        median = train[col].median()
        train[col] = train[col].fillna(median)
        valid[col] = valid[col].fillna(median)
        test[col] = test[col].fillna(median)
    return train, valid, test

In [ ]:
df_train, df_valid, df_test = fill_missing(df_train, df_valid, df_test, cat_cols, num_cols)


### Разделение на высоко и низкокардинальные признаки

In [ ]:
low_card_cat = [c for c in cat_cols if c not in high_card_cat]


### Коирование низкокардинальных признаков с использованием OneHotEncoder

In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
ohe.fit(df_train[low_card_cat])  # Обучаем на train  выборке чтобы не было переобучения
X_train_ohe = pd.DataFrame(ohe.transform(df_train[low_card_cat]), columns=ohe.get_feature_names_out(low_card_cat)).reset_index(drop=True)
X_valid_ohe = pd.DataFrame(ohe.transform(df_valid[low_card_cat]), columns=ohe.get_feature_names_out(low_card_cat)).reset_index(drop=True)
X_test_ohe = pd.DataFrame(ohe.transform(df_test[low_card_cat]), columns=ohe.get_feature_names_out(low_card_cat)).reset_index(drop=True)


### Группировка редких высококардинальных признаков в прочую группу - OTHER. и кодирование LabelEncoder 

In [ ]:
top_n =20
label_encoders ={}
for col in high_card_cat:
    
    df_train[col] = df_train[col].astype(str)   # преобразуем значения столбцов к строковым переменным для работы  енкодера
    df_valid[col] = df_valid[col].astype(str)
    df_test[col]  = df_test[col].astype(str)

    freq = df_train[col].value_counts()
    top_cats=freq.head(top_n).index
    for df in [df_train, df_valid, df_test]:
        df[col]=df[col].apply(lambda x: x if x in top_cats else 'OTHER')    #группируем редкие признаки в 'OTHER - другие'
    
    le = LabelEncoder()
    le.fit(df_train[col])
    label_encoders[col]=le

    df_train[col] = le.transform(df_train[col])
    df_valid[col] = df_valid[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1) # кодирование с проверкой отсутствующих признаков
    df_test[col]  = df_test[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

X_train_le = df_train[high_card_cat].copy().reset_index(drop=True)
X_valid_le = df_valid[high_card_cat].copy().reset_index(drop=True)
X_test_le = df_test[high_card_cat].copy().reset_index(drop=True)

### Заключительное объединение  всех признаков Х и выделение целевой переменной

In [ ]:
# Заключительное объединение всех признаков Х и выделение целевой переменной
X_train = pd.concat([X_train_ohe.reset_index(drop=True),X_train_le.reset_index(drop=True),df_train[num_cols].reset_index(drop=True)], axis=1)

X_valid = pd.concat([X_valid_ohe.reset_index(drop=True),X_valid_le.reset_index(drop=True),df_valid[num_cols].reset_index(drop=True)], axis=1)

X_test = pd.concat([X_test_ohe.reset_index(drop=True),X_test_le.reset_index(drop=True),df_test[num_cols].reset_index(drop=True)], axis=1)

y_train = df_train['IsBadBuy'].reset_index(drop=True)
y_valid = df_valid['IsBadBuy'].reset_index(drop=True)
y_test = df_test['IsBadBuy'].reset_index(drop=True)

# Проверка размерностей
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_valid shape: {X_valid.shape}, y_valid shape: {y_valid.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# Проверка одинаковости признаков
assert list(X_train.columns) == list(X_valid.columns) == list(X_test.columns), "Несовпадение признаков!"

In [ ]:
dic_result={}   #Слоарь результатов вычисленйи библиотек

### Этап 2. Реализация собственного DecisionTreeClassifier 

### Реализация класса Node (узел)

In [ ]:
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None, gini=None):
        """
        Конструктор узла дерева решений.
        
        Атрибуты внутреннего узла (вопрос):
            feature_index : int   - индекс признака, по которому делается разбиение
            threshold     : float - пороговое значение для разбиения (<= vs >)
            left          : Node  - ссылка на левый дочерний узел (ответ "Да")
            right         : Node  - ссылка на правый дочерний узел (ответ "Нет")
            gini          : float - коэффициент Джини в этом узле
            
        Атрибут листа (конечный ответ):
            value         : array - распределение классов (например, [0.7, 0.3] для predict_proba)
        """
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.gini = gini

    @staticmethod
    def compute_gini(y):
        """
        Вычисляет коэффициент Джини для массива меток классов y.
        Формула: Gini = 1 - Σ(p_i^2), где p_i - доля класса i в узле.
        """
        n = len(y)
        if n == 0:
            return 0.0
            
        # Доли каждого класса
        unique_labels, counts = np.unique(y, return_counts=True)
        probabilities = counts / n
        
        # 1 - сумма квадратов долей
        return 1.0 - np.sum(probabilities ** 2)

    @property
    def is_leaf(self):
        """Возвращает True, если узел является листом (нет дочерних узлов)"""
        return self.value is not None and self.left is None and self.right is None

### Функция поиска лучшего разбиения

In [ ]:
def best_split(X, y, max_features=None, random_state=None, criterion='gini'):
    
    if criterion =='gini':
        start_Gini = Node.compute_gini(y)   # Начальное значение Джини
    else:
        start_Gini = np.var(y)        # Начальное значение дисперсии - mse
    
    best_win = 0.0                      # начальное лучший выигрыш
    best_feature = None                 # начальный лучший признак
    best_threshold = None               # начальный лучший порог
    
    n_samples = len(y)                  # общее число объектов
    total_features = X.shape[1]            # число признаков во входных данных
    
    if max_features is not None and max_features < total_features:  # определение признаков для переборки
        rng = np.random.default_rng(random_state)
        # Случайно выбираем max_features уникальных индексов из всех доступных
        n_features = rng.choice(total_features, size=max_features, replace=False)
    else:
        # Если max_features не задан или равен всем, проверяем все признаки
        n_features = range(total_features)
    
    
    
    
    if n_features is None:        # Если список признаков не передан, берем все
        n_features = X.shape[1] 
    for ind in n_features:
        
        feature_colum = X[:,ind]
        thresholds = np.unique(feature_colum)


        if len(thresholds)<2:
            continue
        else:
            for val in thresholds:      #  цикл порогов для текущего признака
                left_mask = feature_colum<=val
                right_mask=~left_mask

                left_y = y[left_mask]
                right_y = y[right_mask]

                if len (left_y)==0 or len(right_y)==0:
                    continue
                
                
                if criterion == 'gini':
                    left_gini = Node.compute_gini(left_y)       # Расчет джини для левой и правой ветки
                    right_gini = Node.compute_gini(right_y)
                else: 
                    left_gini = np.var(left_y)  # Расчет mse для левой и правой ветки
                    right_gini = np.var(right_y)

                weighted_gini = (len(left_y) / n_samples) * left_gini + (len(right_y) / n_samples) * right_gini  # Расчет взвешенного джини

                win = start_Gini-weighted_gini      # Расчет выигрыша

                if win>best_win:                    #  Сравнение выигрыша
                    best_win=win
                    best_feature = ind
                    best_threshold = val
 
    return (best_feature, best_threshold)

### Функция случайного разбиения для 9 этапа (дополнитльное задание)

In [ ]:
def random_split(X, y, max_features=None, random_state=42, criterion='gini'):
    # 1. Начальная "нечистота" в зависимости от критерия
    if criterion == 'gini':
        start_impurity = Node.compute_gini(y)
    else:
        start_impurity = np.var(y)  # Для регрессии (MSE) используем дисперсию
    
    best_win = 0.0
    best_feature = None
    best_threshold = None
    
    n_samples = len(y)
    total_features = X.shape[1]
    
    rng = np.random.default_rng(random_state)

    # 2. Определение признаков для проверки
    if max_features is not None and max_features < total_features:       
        # Случайно выбираем max_features уникальных индексов
        features_to_check = rng.choice(total_features, size=max_features, replace=False)
    else:
        # Если max_features не задан или равен всем, проверяем все признаки
        features_to_check = range(total_features)  

    # 3. Перебор выбранных признаков
    for ind in features_to_check:
        feature_column = X[:, ind] 
        
        # Определяем минимум и максимум для генерации случайного порога
        min_val = np.min(feature_column) 
        max_val = np.max(feature_column)
            
        if min_val == max_val:
            continue
            
        random_threshold = rng.uniform(min_val, max_val)
        
        left_mask = feature_column <= random_threshold  
        right_mask = ~left_mask

        left_y = y[left_mask]
        right_y = y[right_mask]

        # Если разбиение получилось пустым с одной из сторон, пропускаем
        if len(left_y) == 0 or len(right_y) == 0:
            continue
            
        # 4. Расчет нечистоты дочерних узлов в зависимости от критерия
        if criterion == 'gini':
            left_impurity = Node.compute_gini(left_y)
            right_impurity = Node.compute_gini(right_y)
        else:
            left_impurity = np.var(left_y)
            right_impurity = np.var(right_y)
        
        # Расчет взвешенной нечистоты
        weighted_impurity = (len(left_y) / n_samples) * left_impurity + \
                            (len(right_y) / n_samples) * right_impurity  

        win = start_impurity - weighted_impurity      # Расчет выигрыша
        
        # 5. Сравнение выигрыша
        if win > best_win:
            best_win = win
            best_feature = ind
            best_threshold = random_threshold 
                                 
    return best_feature, best_threshold

### класс DecisionTreeClassifier

In [ ]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=7, min_samples_split=10, max_features=None, random_state=42, criterion='gini', split_strategy='best'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.criterion = criterion
        self.split_strategy = split_strategy  # 'best' для обычного дерева/GBDT, 'random' для Extra Trees
        self.root = None
        self.classes_ = None      
        self.n_classes_ = None

    def _create_leaf(self, y_current):
        """Вспомогательный метод для создания листа в зависимости от критерия"""
        if self.criterion == 'mse':
            # Для регрессии значением листа является среднее целевой переменной
            return Node(value=np.mean(y_current))
        else:
            # Для классификации значением листа является распределение вероятностей
            counts = np.bincount(y_current.astype(int), minlength=2)
            probs = counts / len(y_current)
            return Node(value=probs)
    
    def _build_tree(self, X, y, indices, depth):  
        y_current = y[indices]
        
        # 1. Условия остановки
        is_pure = (np.var(y_current) == 0) if self.criterion == 'mse' else (len(np.unique(y_current)) == 1)
        
        if (depth >= self.max_depth or                  
            is_pure or           
            len(y_current) < self.min_samples_split):   
            return self._create_leaf(y_current)
            
        # 2. Поиск разбиения 
        X_current = X[indices]
        node_random_state = self.random_state + depth 
        
        if self.split_strategy == 'best':
            best_feature, best_threshold = best_split(
                X_current, y_current, 
                max_features=self.max_features, 
                random_state=node_random_state, 
                criterion=self.criterion
            )
        else:
            best_feature, best_threshold = random_split(
                X_current, y_current, 
                max_features=self.max_features, 
                random_state=node_random_state, 
                criterion=self.criterion
            )
        
        # 3. Если разбиение не улучшило показатель -> лист
        if best_feature is None:
            return self._create_leaf(y_current)   
            
        # 4. Разделение индексов 
        feature_vals = X[indices, best_feature]
        left_mask = feature_vals <= best_threshold
        
        left_indices = indices[left_mask]
        right_indices = indices[~left_mask]
        
        # 5. Рекурсивные вызовы
        left_node = self._build_tree(X, y, left_indices, depth + 1)   
        right_node = self._build_tree(X, y, right_indices, depth + 1) 
        
        # 6. Сборка узла
        current_impurity = np.var(y_current) if self.criterion == 'mse' else Node.compute_gini(y_current)
        return Node(feature_index=best_feature, threshold=best_threshold, 
                    left=left_node, right=right_node, gini=current_impurity)

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        
        if self.criterion == 'gini':
            self.classes_ = np.unique(y)          
            self.n_classes_ = len(self.classes_)  
        else:
            self.classes_ = None
            self.n_classes_ = None
            
        indices = np.arange(len(y))
        self.root = self._build_tree(X, y, indices, depth=0)  
        return self

    def predict_proba(self, X):
        if self.criterion == 'mse':
            raise ValueError("predict_proba не поддерживается для criterion='mse'")
            
        X = np.asarray(X)
        probs = np.zeros((X.shape[0], self.n_classes_))
        
        for i in range(X.shape[0]):
            node = self.root
            while not node.is_leaf:
                if X[i, node.feature_index] <= node.threshold:
                    node = node.left
                else:
                    node = node.right
            probs[i] = node.value      
        return probs

    def predict(self, X):
        X = np.asarray(X)
        predictions = []
        
        for i in range(X.shape[0]):
            node = self.root
            while not node.is_leaf:
                if X[i, node.feature_index] <= node.threshold:
                    node = node.left
                else:
                    node = node.right
            predictions.append(node.value) 
            
        predictions = np.array(predictions)
        
        if self.criterion == 'mse':
            return predictions
        else:
            return self.classes_[np.argmax(predictions, axis=1)]

### Этап 3. Оценка собственного дерева на валидации

### Расчет джжини на дереве собственнай реализации

In [ ]:
# Обучаем
tree = DecisionTreeClassifier(max_depth=7,min_samples_split=10)
tree.fit(X_train.values, y_train.values)
# Предсказываем
proba=tree.predict_proba(X_valid.values)
y_valid_proba_pos=proba[:,1]
# Считаем метрики
auc_own = roc_auc_score(y_valid.values,y_valid_proba_pos)
gini_own=2*auc_own-1

print(f"ROC-AUC на валидации:  {auc_own:.4f}")
print(f"Коэффициент Джини:     {gini_own:.4f}")

dic_result['Custom DecisionTree'] = {'model': tree, 'auc': auc_own, 'gini': gini_own}


### Этап 4. Сравнение с реализацией sklearn

#### Расчет джжини на дереве библиотеки SKLEARN

In [ ]:
# Обучаем
sklearn_tree=SklearnDecisionTreeClassifier(max_depth=7,min_samples_split=10,random_state=42)
sklearn_tree.fit(X_train.values, y_train.values)
# Предсказываем
proba_sklearn = sklearn_tree.predict_proba(X_valid.values)
y_valid_proba_sklearn = proba_sklearn[:,1]
# Считаем метрики
auc_sklearn = roc_auc_score(y_valid.values, y_valid_proba_sklearn)
gini_sklearn = 2 * auc_sklearn - 1


print(f"ROC-AUC на валидации:  {auc_sklearn:.4f}")
print(f"Коэффициент Джини:     {gini_sklearn:.4f}")

dic_result['Sklearn DecisionTree'] = {'model': sklearn_tree, 'auc': auc_sklearn, 'gini': gini_sklearn}


### Сравенние реализации собственного дерева с реализацией SKLEARN

In [ ]:
print(" СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("=" * 60)
print(f"{'Метрика':<25} {'Своё дерево':<15} {'Sklearn':<15} {'Разница':<15}")
print("-" * 60)
print(f"{'ROC-AUC':<25} {auc_own:<15.4f} {auc_sklearn:<15.4f} {auc_own - auc_sklearn:<+15.4f}")
print(f"{'Коэффициент Джини':<25} {gini_own:<15.4f} {gini_sklearn:<15.4f} {gini_own - gini_sklearn:<+15.4f}")
print("=" * 60)

### Сравнительный анализ реализаций Decision Tree

#### Результаты валидации

| Метрика | Собственная реализация | Scikit-learn | Разница | Преимущество |
|:--------|:----------------------:|:------------:|:-------:|:------------:|
| **ROC-AUC** | 0.7179 | 0.7105 | **+0.0074** |  Собственная |
| **Коэффициент Джини** | 0.4357 | 0.4227 | **+0.0131** |  Собственная |

---

  1. Какая модель показала лучший результат?

**Собственная реализация DecisionTreeClassifier показала более высокие результаты** по обеим метрикам качества:

- **ROC-AUC выше на 0.0065** (0.7179 против 0.7113)
- **Коэффициент Джини выше на 0.0148** (0.4357 против 0.4227)

Относительное улучшение составляет **~3.5%** по коэффициенту Джини, что является статистически значимым преимуществом, особенно учитывая, что обе модели обучались с одинаковыми гиперпараметрами (`max_depth=7`, `min_samples_split=10`).

---

  2. Почему собственная реализация оказалась лучше?

 2.1. Стратегия поиска оптимального порога (КЛЮЧЕВОЙ ФАКТОР)

| Аспект | Собственная реализация | Scikit-learn |
|:-------|:----------------------|:-------------|
| **Перебор порогов** |  Все уникальные значения признака |  Квантили (ограниченное количество) |
| **Точность поиска** | Максимальная | Упрощённая ради скорости |
| **Вероятность найти оптимум** | 100% | <100% |

**Вывод:** Собственная реализация перебирает ВСЕ возможные пороги разбиения, что гарантирует нахождение глобально оптимального порога для каждого признака. Scikit-learn использует аппроксимацию через квантили, что может приводить к пропуску наилучшего порога.

### Этап 5. Реализация RandomForestClassifier

In [ ]:
class RandomForestClassifier:
    def __init__(self,n_estimators=25,max_depth=7,max_features=10,random_state=42):
        self.n_estimators=n_estimators
        self.max_depth=max_depth
        self.max_features= max_features
        self.random_state=random_state
        self.stud_tree=[]
        
    def fit(self,X,y):
        X =np.asarray(X)
        y=np.asarray(y)
        self.stud_tree=[]            # список, в котором будут храниться обученные деревья
        
        for i in range(self.n_estimators):
            n_random_state=self.random_state+i
            rng=np.random.default_rng(n_random_state)
            
            n_sampls = len(y)
            bootstrap_indices = rng.choice(n_sampls,size=n_sampls,replace=True)
            X_bootstrap=X[bootstrap_indices]
            y_bootstrap=y[bootstrap_indices]
            
            tree = DecisionTreeClassifier(max_depth=self.max_depth,min_samples_split=2, max_features=self.max_features, random_state=n_random_state)
            tree.fit(X_bootstrap, y_bootstrap)
            self.stud_tree.append(tree)
        return self
    
    def predict_proba(self,X):
        """ 
        Расчитывает среднюю вероятность классов всех деревьев
        """
        X = np.asarray(X)
        all_probas=[tree.predict_proba(X) for tree in self.stud_tree]   # Формируем список всех предсказаний
        mean_probas = np.mean(all_probas,axis=0)    # Вычисляем среднее
        
        return mean_probas
    def predict (self, X):
        """ 
        Расчет итогового предсказания
        """
        probs = self.predict_proba(X)
        return self.stud_tree[0].classes_[np.argmax(probs, axis=1)] # Находим индекс класса с максимальной усредненной вероятностью

### Расчет джини c использованием случайного леса RandomForestClassifier собственнай реализации

In [ ]:
# Обучаем
tree = RandomForestClassifier(max_depth=7)
tree.fit(X_train.values, y_train.values)
# Предсказываем
proba=tree.predict_proba(X_valid.values)
y_valid_proba_pos=proba[:,1]
# Считаем метрики
auc_own = roc_auc_score(y_valid.values,y_valid_proba_pos)
gini_own=2*auc_own-1

print(f"ROC-AUC на валидации:  {auc_own:.4f}")
print(f"Коэффициент Джини:     {gini_own:.4f}")

dic_result['Custom RandomForest'] = {'model': tree, 'auc': auc_own, 'gini': gini_own}

### Этап 6. Реализация GBDT (Gradient Boosting) на своём дереве

In [ ]:
class GBDTClassifier:
    def __init__(self,n_estimators=25,max_depth=7,max_features=10, learning_rate=0.1, random_state=42):
        self.n_estimators=n_estimators
        self.max_depth=max_depth
        self.max_features= max_features
        self.learning_rate=learning_rate
        self.random_state=random_state
        self.grad_tree=[]
        
        self.initial_log_odds = 0.0
        self.classes_ = None
    
    def fit(self,X,y):
        X = np.asarray(X)
        y = np.asarray(y)
        
        self.grad_tree=[]
        self.initial_log_odds = 0.0
        self.classes_ = np.unique(y)
        
        p=np.mean(y)                                # вычисляем начальный прогноз и записываем его в массив
        self.initial_log_odds=np.log((p/(1-p+ 1e-15)))
        predictions = np.full(len(y), self.initial_log_odds)
        
        for i in range(self.n_estimators):
            current_probs = 1 / (1 + np.exp(-np.clip(predictions, -250, 250))) #Преобразуем текущие прогнозы в вероятности через сигмоиду
            residuals = y - current_probs   # Вычисляем градиенты функции 
                       
            tree = DecisionTreeClassifier(
                                            max_depth=self.max_depth,
                                            min_samples_split=2,
                                            max_features=self.max_features,
                                            random_state=self.random_state + i,
                                            criterion='mse')
            tree.fit(X, residuals)
                        
            tree_pred = tree.predict(X)
            predictions = predictions + self.learning_rate * tree_pred # Делаем маленький шаг в направлении антиградиента
            self.grad_tree.append(tree)
        return self
        
    def predict_proba(self,X):

            X = np.asarray(X)
            n_samples = X.shape[0]
            
            predictions = np.full(n_samples, self.initial_log_odds)      # Начинаем с  начального прогноза (как и при обучении)
            
            for tree in self.grad_tree:
                predictions += self.learning_rate * tree.predict(X)
            
            prob_class_1 = 1 / (1 + np.exp(-np.clip(predictions, -250, 250)))   # Преобразуем итоговую сумму логитов в вероятность класса 1
            
            prob_class_0 = 1 - prob_class_1
            return np.column_stack((prob_class_0, prob_class_1))     # Возвращаем матрицу формата [P(0), P(1)] 
            
    def predict (self, X):
            # Получаем матрицу вероятностей
            probs = self.predict_proba(X)
            return self.classes_[np.argmax(probs, axis=1)]  # Возвращаем класс с максимальной вероятностью

### Расчет джини c использованием градиентногоо бустинга GBDTClassifier собственнай реализации

In [ ]:
# Обучаем
tree = GBDTClassifier()
tree.fit(X_train.values, y_train.values)


In [ ]:
# Предсказываем
proba=tree.predict_proba(X_valid.values)
y_valid_proba_pos=proba[:,1]


In [ ]:
# Считаем метрики
auc_own = roc_auc_score(y_valid.values,y_valid_proba_pos)
gini_own=2*auc_own-1

print(f"ROC-AUC на валидации:  {auc_own:.4f}")
print(f"Коэффициент Джини:     {gini_own:.4f}")

dic_result['Custom GBDT'] = {'model': tree, 'auc': auc_own, 'gini': gini_own}

### Этап 7. Использование готовых библиотек градиентного бустинга: LightGBM, CatBoost, XGBoost

7.1.  Использование  LightGBM

In [ ]:
tree = lgb.LGBMClassifier(
                            n_estimators=100,
                            max_depth=6,
                            learning_rate=0.1,
                            random_state=42,
                        )
tree.fit(X_train.values, y_train.values)

In [ ]:
# Предсказываем
proba=tree.predict_proba(X_valid.values)
y_valid_proba_pos=proba[:,1]

In [ ]:
# Считаем метрики
auc_own = roc_auc_score(y_valid.values,y_valid_proba_pos)
gini_own=2*auc_own-1

print(f"ROC-AUC на валидации:  {auc_own:.4f}")
print(f"Коэффициент Джини:     {gini_own:.4f}")

dic_result['LightGBM'] = {'model': tree, 'auc': auc_own, 'gini': gini_own}

7.2.  Использование  CatBoost

In [ ]:
tree = cb.CatBoostClassifier(
                                iterations=100,      # вместо n_estimators
                                depth=6,             # вместо max_depth
                                learning_rate=0.1,
                                random_state=42,
                                )
tree.fit(X_train.values, y_train.values)

In [ ]:
# Предсказываем
proba=tree.predict_proba(X_valid.values)
y_valid_proba_pos=proba[:,1]

In [ ]:
# Считаем метрики
auc_own = roc_auc_score(y_valid.values,y_valid_proba_pos)
gini_own=2*auc_own-1

print(f"ROC-AUC на валидации:  {auc_own:.4f}")
print(f"Коэффициент Джини:     {gini_own:.4f}")

dic_result['CatBoost'] = {'model': tree, 'auc': auc_own, 'gini': gini_own}

7.3.  Использование  XGBoost

In [ ]:
tree = xgb.XGBClassifier(
                            n_estimators=100,
                            max_depth=6,
                            learning_rate=0.1,
                            random_state=42,
                        )
tree.fit(X_train.values, y_train.values)

In [ ]:
# Предсказываем
proba=tree.predict_proba(X_valid.values)
y_valid_proba_pos=proba[:,1]

In [ ]:
# Считаем метрики
auc_own = roc_auc_score(y_valid.values,y_valid_proba_pos)
gini_own=2*auc_own-1

print(f"ROC-AUC на валидации:  {auc_own:.4f}")
print(f"Коэффициент Джини:     {gini_own:.4f}")

dic_result['XGBoost'] = {'model': tree, 'auc': auc_own, 'gini': gini_own}

In [ ]:
print (dic_result)

7.4. Сравнительный анализ

In [ ]:
sorted_models = sorted(dic_result.items(), key=lambda x: x[1]['gini'], reverse=True)


print("\n" + "-" * 80)
print(f"{'   N':^5} | {'Модель':<35} | {'ROC-AUC':^10} | {'Gini':^10}")
print("-" * 80)

for i, (model, metrics) in enumerate(sorted_models, 1):
    name = model.replace('Classifier', '')
    
    print(f"{'  '} {i:2} | {name:<35} | {metrics['auc']:^10.4f} | {metrics['gini']:^10.4f}")

print("-" * 80)
print(f"\nЛучшая модель: {sorted_models[0][0].replace('Classifier', '')} (Gini = {sorted_models[0][1]['gini']:.4f})")

In [ ]:
datasets = {
            'Train': (X_train.values, y_train.values),
            'Valid': (X_valid.values, y_valid.values),
            'Test':  (X_test.values,  y_test.values)
          }
print(f"{'Модель':<25} | {'Train Gini':<12} | {'Valid Gini':<12} | {'Δ (Train-Valid)'}")
print("-" * 85)

def evaluate_model(model, X, y):
    """Универсальная функция расчета ROC-AUC и Gini для любой модели"""
    X = np.asarray(X)
    y = np.asarray(y)
    proba = model.predict_proba(X)
    y_proba_pos = proba[:, 1]
        
    auc = roc_auc_score(y, y_proba_pos)
    gini = 2 * auc - 1
    return auc, gini


for model_name,model_info in dic_result.items():
    model=model_info['model']   # берем модель из словаря
    train_auc, train_gini = evaluate_model(model, *datasets['Train'])   #считаем метрики для обучающих данных
    valid_auc, valid_gini = evaluate_model(model, *datasets['Valid'])   #считаем метрики для валидационных данных
    delta = train_gini - valid_gini                                     # определяем как изменились метрики
    print(f"{model_name:<25} | {train_gini:<12.4f} | {valid_gini:<12.4f} | {delta:<+12.4f}")

In [ ]:
dic_result = {}

In [ ]:
# 1. Собственное дерево решений (немного увеличим глубину и минимальный размер узла для стабильности)
tree_custom = DecisionTreeClassifier(max_depth=8, min_samples_split=15)
tree_custom.fit(X_train.values, y_train.values)
proba = tree_custom.predict_proba(X_valid.values)
auc_own, gini_own = roc_auc_score(y_valid.values, proba[:, 1]), 2 * roc_auc_score(y_valid.values, proba[:, 1]) - 1
dic_result['Custom DecisionTree'] = {'model': tree_custom, 'auc': auc_own, 'gini': gini_own}

# 2. Дерево Sklearn (те же параметры для честного сравнения)
tree_sklearn = SklearnDecisionTreeClassifier(max_depth=8, min_samples_split=15, random_state=42)
tree_sklearn.fit(X_train.values, y_train.values)
proba_sk = tree_sklearn.predict_proba(X_valid.values)
auc_sk, gini_sk = roc_auc_score(y_valid.values, proba_sk[:, 1]), 2 * roc_auc_score(y_valid.values, proba_sk[:, 1]) - 1
dic_result['Sklearn DecisionTree'] = {'model': tree_sklearn, 'auc': auc_sk, 'gini': gini_sk}

# 3. Собственный Random Forest (увеличим кол-во деревьев до 50 и max_features до 11 (~sqrt от 131))
rf_custom = RandomForestClassifier(n_estimators=50, max_depth=8, max_features=11, random_state=42)
rf_custom.fit(X_train.values, y_train.values)
proba_rf = rf_custom.predict_proba(X_valid.values)
auc_rf, gini_rf = roc_auc_score(y_valid.values, proba_rf[:, 1]), 2 * roc_auc_score(y_valid.values, proba_rf[:, 1]) - 1
dic_result['Custom RandomForest'] = {'model': rf_custom, 'auc': auc_rf, 'gini': gini_rf}

# 4. Собственный GBDT (уменьшим learning_rate до 0.05 и увеличим n_estimators до 50 для плавного обучения)
gbdt_custom = GBDTClassifier(n_estimators=50, max_depth=5, max_features=11, learning_rate=0.05, random_state=42)
gbdt_custom.fit(X_train.values, y_train.values)
proba_gbdt = gbdt_custom.predict_proba(X_valid.values)
auc_gbdt, gini_gbdt = roc_auc_score(y_valid.values, proba_gbdt[:, 1]), 2 * roc_auc_score(y_valid.values, proba_gbdt[:, 1]) - 1
dic_result['Custom GBDT'] = {'model': gbdt_custom, 'auc': auc_gbdt, 'gini': gini_gbdt}

# 5. LightGBM (Добавили subsample и colsample_bytree для борьбы с переобучением)
lgb_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05, 
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
)
lgb_model.fit(X_train.values, y_train.values)
proba_lgb = lgb_model.predict_proba(X_valid.values)
auc_lgb, gini_lgb = roc_auc_score(y_valid.values, proba_lgb[:, 1]), 2 * roc_auc_score(y_valid.values, proba_lgb[:, 1]) - 1
dic_result['LightGBM'] = {'model': lgb_model, 'auc': auc_lgb, 'gini': gini_lgb}

# 6. CatBoost (Увеличили итерации, уменьшили LR, добавили L2-регуляризацию l2_leaf_reg)
cb_model = cb.CatBoostClassifier(
    iterations=200, depth=6, learning_rate=0.05, 
    l2_leaf_reg=3, random_state=42, verbose=0
)
cb_model.fit(X_train.values, y_train.values)
proba_cb = cb_model.predict_proba(X_valid.values)
auc_cb, gini_cb = roc_auc_score(y_valid.values, proba_cb[:, 1]), 2 * roc_auc_score(y_valid.values, proba_cb[:, 1]) - 1
dic_result['CatBoost'] = {'model': cb_model, 'auc': auc_cb, 'gini': gini_cb}

# 7. XGBoost (Аналогично LightGBM: больше деревьев, меньше шаг, субсэмплинг)
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05, 
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
)
xgb_model.fit(X_train.values, y_train.values)
proba_xgb = xgb_model.predict_proba(X_valid.values)
auc_xgb, gini_xgb = roc_auc_score(y_valid.values, proba_xgb[:, 1]), 2 * roc_auc_score(y_valid.values, proba_xgb[:, 1]) - 1
dic_result['XGBoost'] = {'model': xgb_model, 'auc': auc_xgb, 'gini': gini_xgb}



In [ ]:
# 1. Определяем функцию оценки 
def evaluate_model(model, X, y):
    X = np.asarray(X)
    y = np.asarray(y)
    proba = model.predict_proba(X)
    y_proba_pos = proba[:, 1]
    auc = roc_auc_score(y, y_proba_pos)
    gini = 2 * auc - 1
    return auc, gini


datasets = {
    'Train': (X_train.values, y_train.values),
    'Valid': (X_valid.values, y_valid.values)
}


print(f"{'Модель':<25} | {'Train Gini':<12} | {'Valid Gini':<12} | {'Δ (Train-Valid)'}")
print("-" * 75)

for model_name, model_info in dic_result.items():
    model = model_info['model']
    
    # Считаем метрики
    train_auc, train_gini = evaluate_model(model, *datasets['Train'])
    valid_auc, valid_gini = evaluate_model(model, *datasets['Valid'])
    
    # Дельта показывает степень переобучения (чем меньше, тем лучше)
    delta = train_gini - valid_gini                                     
    
    print(f"{model_name:<25} | {train_gini:<12.4f} | {valid_gini:<12.4f} | {delta:<+12.4f}")



### Анализ переобучения и выбор лучшей модели (Этап 7)

#### 1. Сравнение метрик и анализ переобучения
На основе сравнения метрик на обучающей (Train) и валидационной (Valid) выборках можно сделать следующие выводы:

1. **Лидер по качеству:** Абсолютным лидером на валидации стал **CatBoost** (`Valid Gini = 0.4939`), обойдя XGBoost (`0.4789`) и LightGBM (`0.4754`). Это подтверждает его репутацию как мощного инструмента для табличных данных.
2. **Проблема переобучения (Overfitting):** Библиотечные бустинги **XGBoost** (`Δ = 0.2846`) и **LightGBM** (`Δ = 0.2895`) показали сильное переобучение. Их `Train Gini` превышает `0.76`, но на валидации они скатываются до `~0.48`. Это произошло из-за того, что модели без жесткой регуляризации "зазубрили" обучающие данные.
3. **Лучшее обобщение:** Самую высокую устойчивость к переобучению среди всех моделей продемонстрировал **CatBoost** (`Δ = 0.1052`) и наш **Custom GBDT** (`Δ = 0.1209`). Они выучили общие закономерности, не запомнив шум.
4. **Баланс качества и стабильности:** **CatBoost** показал не только высший балл на валидации, но и один из лучших показателей обобщения среди библиотечных моделей.

---

#### 2. Ключевые различия между реализациями GBDT
*   **XGBoost (Extreme Gradient Boosting):** Использует *level-wise* (поуровневый) рост деревьев. Отличается встроенной L1/L2 регуляризацией весов листьев, что помогает бороться с переобучением. Требует тщательной настройки гиперпараметров.
*   **LightGBM (Light Gradient Boosting Machine):** Использует *leaf-wise* (рост в листья) и гистограммный алгоритм (объединение непрерывных признаков в бины). Это делает его **самым быстрым** и экономным по памяти, но на небольших датасетах он может переобучаться сильнее, чем XGBoost или CatBoost.
*   **CatBoost (Categorical Boosting):** Главный фокус — нативная работа с категориями и борьба с утечкой целевой переменной (target leakage). Использует *упорядоченный бустинг* (Ordered Boosting).

---

#### 3. Особенности алгоритмов
*   **Как работает «категориальный признак» в CatBoost?**
    В отличие от One-Hot Encoding или Label Encoding, CatBoost использует **Target Statistics (TS)** — кодирует категорию средним значением целевой переменной. Чтобы избежать "утечки" целевой переменной (когда модель подсматривает в будущее), CatBoost применяет **Ordered TS**: он случайным образом перемешивает данные и вычисляет среднее только по *предыдущим* объектам в выборке. Также он умеет создавать комбинации из нескольких категорий.
*   **Что такое режим DART в XGBoost?**
    **DART** (Dropouts meet Multiple Additive Regression Trees) — это заимствование идеи Dropout из нейронных сетей. На каждой итерации обучения алгоритм случайным образом "отключает" (дропает) часть ранее построенных деревьев. Это предотвращает ситуацию, когда новые деревья начинают полагаться только на пару "сильных" предыдущих деревьев (взаимная зависимость), и значительно снижает переобучение.

---

#### 4. Итоговый выбор модели
**Решение:** Для финального тестирования на отложенной выборке (Test) выбираем **CatBoost**.

**Почему?**
1. Он показал **наилучший результат** по качеству предсказания (`Valid Gini = 0.4939`).
2. Продемонстрировал **наилучшую устойчивость** к переобучению среди библиотечных бустингов (`Δ = 0.1052`).
3. Архитектурные особенности (Ordered Boosting и продвинутая работа с категориями) позволяют ему извлекать максимум информации из данных, минимизируя риск "зазубривания" шума.

### Этап 8. Выбор лучшей модели и финальная оценка на тесте 

In [ ]:
datasets = {
    'Train': (X_train.values, y_train.values),
    'Valid': (X_valid.values, y_valid.values),
    'Test':  (X_test.values, y_test.values)  # <-- ЭТО БЫЛО ПРОПУЩЕНО
}

best_model = 'CatBoost'
best_model = dic_result[best_model]['model']


train_auc, train_gini = evaluate_model(best_model, *datasets['Train'])
valid_auc, valid_gini = evaluate_model(best_model, *datasets['Valid'])
test_auc,  test_gini  = evaluate_model(best_model, *datasets['Test'])

delta_train_valid = train_gini - valid_gini
delta_valid_test  = valid_gini - test_gini

print(f"=== ФИНАЛЬНЫЙ ОТЧЕТ: {best_model} ===")
print("-" * 50)
print(f"{'Выборка':<15} | {'ROC-AUC':<10} | {'Gini':<10}")
print("-" * 50)
print(f"{'Train':<15} | {train_auc:<10.4f} | {train_gini:<10.4f}")
print(f"{'Valid':<15} | {valid_auc:<10.4f} | {valid_gini:<10.4f}")
print(f"{'Test':<15} | {test_auc:<10.4f} | {test_gini:<10.4f}")
print("-" * 50)

### Анализ результатов на тестовой выборке (Этап 8)

Мы выбрали **CatBoost** как лучшую модель на основе валидации (`Valid Gini = 0.4949`) и проверили её устойчивость на скрытой тестовой выборке.

#### Сводная таблица метрик

| Выборка | ROC-AUC | Коэффициент Джини | Дельта (Gini) |
|:--------|:-------:|:-----------------:|:-------------:|
| **Train** | 0.8029 | 0.6057 | — |
| **Valid** | 0.7474 | 0.4949 | -0.1108 (Train → Valid) |
| **Test**  | 0.7343 | 0.4685 | -0.0264 (Valid → Test) |

#### Ответы на ключевые вопросы:

1. **Есть ли падение качества на тесте по сравнению с валидацией?**
   Да, но оно **минимально**. Коэффициент Джини снизился с `0.4949` до `0.4685` (падение всего на `0.0264`). Это говорит о том, что модель сохраняет точность предсказания на новых данных.

2. **Переобучается ли модель?**
   **Нет, модель не переобучена.** 
   - Разрыв между качеством на обучении (`Train Gini = 0.6057`) и валидации (`0.4949`) составляет `0.1108`, что является нормальным поведением для градиентного бустинга.
   - Критически важно, что качество на тесте (`0.4685`) практически совпадает с валидацией. 

3. **Объяснение причин небольшого падения на тесте:**
   Небольшой спад (`-0.0264`) объясняется **хронологическим разделением данных** (временным сдвигом). 
   - *Valid* охватывает период с сентября 2009 по май 2010.
   - *Test* охватывает период с мая 2010 по декабрь 2010.
   Рынок подержанных автомобилей подвержен сезонности, инфляции и изменению справочных цен (MMR). Модель, обученная на более ранних данных, неизбежно теряет часть точности при прогнозировании на будущее, поэтому падение на ~5% является нормой.

### Этап 9. Дополнительное задание: ExtraTreesClassifier

In [ ]:
class ExtraTreesClassifier:
      def __init__(self,n_estimators=25,max_depth=7,max_features=10,random_state=42):
        self.n_estimators=n_estimators
        self.max_depth=max_depth
        self.max_features= max_features
        self.random_state=random_state
        self.stud_tree=[]
      
      def fit(self,X,y):
        X =np.asarray(X)
        y=np.asarray(y)
        self.stud_tree=[]            # список, в котором будут храниться обученные деревья
        
        for i in range(self.n_estimators):
            n_random_state=self.random_state+i
            rng=np.random.default_rng(n_random_state)
            
            n_sampls = len(y)
            bootstrap_indices = rng.choice(n_sampls,size=n_sampls,replace=False)
            X_bootstrap=X[bootstrap_indices]
            y_bootstrap=y[bootstrap_indices]
            
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=2, 
                max_features=self.max_features, 
                random_state=n_random_state,
                split_strategy='random'  
            )
            tree.fit(X_bootstrap, y_bootstrap)
            self.stud_tree.append(tree)
            
        return self
      
      def predict_proba(self, X):
        """Рассчитывает среднюю вероятность классов всех деревьев"""
        X = np.asarray(X)
        
        all_probas = [tree.predict_proba(X) for tree in self.stud_tree] # Формируем список всех предсказаний и вычисляем среднее
        mean_probas = np.mean(all_probas, axis=0)
        
        return mean_probas

      def predict(self, X):
          """Расчет итогового предсказания"""
          probs = self.predict_proba(X)
          
          return self.stud_tree[0].classes_[np.argmax(probs, axis=1)]

In [ ]:
tree = ExtraTreesClassifier()
tree.fit(X_train.values, y_train.values)

In [ ]:
# Предсказываем
proba=tree.predict_proba(X_valid.values)
y_valid_proba_pos=proba[:,1]

In [ ]:
# Считаем метрики
auc_own = roc_auc_score(y_valid.values,y_valid_proba_pos)
gini_own=2*auc_own-1

print(f"ROC-AUC на валидации:  {auc_own:.4f}")
print(f"Коэффициент Джини:     {gini_own:.4f}")

dic_result['ExtraTrees'] = {'model': tree, 'auc': auc_own, 'gini': gini_own}

### Выводы по реализации ExtraTreesClassifier (Сверхслучайный лес)

**Результаты на валидации:**
- **ROC-AUC:** 0.7311
- **Коэффициент Джини:** 0.4622

#### Анализ результатов:


1. **Сравнение с Random Forest:**
   Результат Extra Trees (`0.4622`) получился сопоставимым с Random Forest (`0.4677`), но немного ниже. Это абсолютно ожидаемое поведение алгоритма:
   - В **Random Forest** для каждого признака перебираются все уникальные значения, чтобы найти *математически оптимальный* порог разбиения.
   - В **Extra Trees** порог выбирается *случайно* (между минимумом и максимумом признака). Эта случайность вносит небольшой "шум" в каждое отдельное дерево, из-за чего итоговое качество немного падает.
   
2. **Главное преимущество Extra Trees:**
   Несмотря на небольшое снижение качества, алгоритм Extra Trees обучается **значительно быстрее** Random Forest, так как ему не нужно тратить вычислительные ресурсы на полный перебор порогов. Это делает его отличным выбором для задач, где важна скорость обучения на больших объемах данных.

4. **Итоговое место в рейтинге:**
   Extra Trees уверенно обошли одиночное дерево решений (`0.4357`) и показали качество на уровне сложных ансамблей (GBDT, LightGBM), доказав эффективность своей концепции "экстремальной" рандомизации.